# Sign Language
Data Analysis - ISAE 2025/2026

In [ ]:
import os
import sys

REPO_URL = "https://github.com/PetitMalo/BE_data_analysis.git"
REPO_NAME = "BE_data_analysis"

if "google.colab" in str(get_ipython()):
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

    os.chdir(REPO_NAME)

    !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] += ":" + os.path.expanduser("~/.cargo/bin")

    print("Installation des dépendances via uv...")
    !uv pip install --system -r pyproject.toml

    repo_path = os.getcwd()
    if repo_path not in sys.path:
        sys.path.append(repo_path)

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.hyper_param_search import find_xgb_hyperparams
from src.utils import logger

%matplotlib inline

Loading the dataset

In [ ]:
dataPath = "./data"
X_file = os.path.join(dataPath, "X.npy")  # X
y_file = os.path.join(dataPath, "y.npy")  # y

## 1 - Data import and formatting


In [ ]:
# Load input data and labels
X = np.load(X_file)
y = np.load(y_file)
print("Shapes of input data:")
print(X.shape)
print("Shapes of labels:")
print(y.shape)

In [ ]:
# Display an image and its label
img_idx = 55
plt.imshow(X[img_idx])
plt.show()
print("Label of image is:")
print(y[img_idx])

In [ ]:
# Let's define a function that converts the one-hot encoded labels to the corresponding sign
# language value.
def label_to_sign_number(label: int) -> int:
    """
    Convert a one-hot encoded label to the
    corresponding sign language value.
    Mapping it with the truth value.

    Args:
      label: (numpy.array) One-hot encoded label

    Returns: (float) Sign language value
    """
    label_value = np.argmax(label)
    mapping = {0: 9, 1: 0, 2: 7, 3: 6, 4: 1, 5: 8, 6: 4, 7: 3, 8: 2, 9: 5}
    return mapping[label_value]

In [ ]:
# Let's test our function
y55 = label_to_sign_number(y[img_idx])
print("Image 55 is a: {}".format(y55))
plt.imshow(X[img_idx])
plt.show()

## 2 - Features construction with filters

In [ ]:
# Let's apply some edge detectors on an example image
img_idx = 900  # select an image index
img = X[img_idx]  # get the grayscale image matrix

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])

gx = ndimage.convolve(img, sobel_x)
gy = ndimage.convolve(img, sobel_y)

edges_sobel = np.sqrt(gx**2 + gy**2)

laplacian = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])

edges_laplacien = ndimage.convolve(img, laplacian)

In [ ]:
# Let's display the image
def displayImg(_img, title=""):
    plt.imshow(_img)
    plt.title(title)
    plt.show()


displayImg(img, "Original Image")
displayImg(edges_sobel, "Sobel gradient Image")
displayImg(edges_laplacien, "Laplacien gradient Image")

In [ ]:
# Train, test, split
X_flat = X.reshape((X.shape[0], -1))  # flatten the images
y_flat = np.array([label_to_sign_number(label) for label in y])
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y_flat, test_size=0.25, random_state=42, stratify=y_flat
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

## 3 - Classification with models

### 3.1 - Ensembling models

#### 3.1.1 - XGBClassifier

In [ ]:
hyper_params = find_xgb_hyperparams(X_train, y_train, X_val, y_val)
logger.info(f"Optimal hyperparameters found: {hyper_params}")

In [ ]:
clf = GradientBoostingClassifier(**hyper_params, random_state=42, verbose=1).fit(X_train, y_train)
clf.score(X_test, y_test)

### 3.2 - MLPClassifier